# Thống kê Quá trình Phân tích Cú pháp & Khử trùng lặp

Notebook này tự động nạp số liệu từ tệp `summaries/parsing_summary.json` và sinh trực tiếp các bảng biểu, đồ thị phân tích trong quá trình biên dịch tài liệu (`jupyter-book build`).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Tìm thư mục gốc của book
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
summary_path = ROOT / 'summaries/parsing_summary.json'

if not summary_path.exists():
    raise FileNotFoundError(f"Không tìm thấy tệp tổng kết tại: {summary_path}")

data = json.loads(summary_path.read_text(encoding='utf-8'))
cov = data['coverage_statistics']
dedup = data['deduplication_statistics']

display(Markdown(f"### Môi trường thực thi: `{data['environment']['platform']}`"))

## 1. Độ bao phủ & Tỷ lệ Phân tích Thành công

Thống kê tổng số bài báo và phiên bản được phân công xử lý trong dải dữ liệu của nhóm.

In [ ]:
cov_df = pd.DataFrame([{
    'Tổng ấn phẩm được giao': cov['total_assigned_publications'],
    'Thành công (có TeX)': cov['successfully_parsed_publications'],
    'Rỗng (PDF-only)': cov['pdf_only_empty_publications'],
    'Tỷ lệ hoàn thành (%)': f"{cov['parsing_success_rate_pct']} %",
    'Tổng số version': cov['total_versions_discovered'],
    'Version parse thành công': cov['parsed_versions_count'],
    'Version PDF-only': cov['empty_versions_count'],
}])
display(cov_df)

# Trực quan hóa tỷ lệ thành công của bài báo và phiên bản
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Biểu đồ tròn cho ấn phẩm
pub_labels = ['Parsed (TeX Source)', 'Empty (PDF-only)']
pub_sizes = [cov['successfully_parsed_publications'], cov['pdf_only_empty_publications']]
pub_colors = ['#2ca02c', '#ff7f0e']
ax1.pie(pub_sizes, labels=pub_labels, autopct='%1.1f%%', startangle=140, colors=pub_colors, explode=(0, 0.1))
ax1.set_title('Tỷ lệ Ấn phẩm theo Nguồn Dữ liệu', fontweight='bold')

# Biểu đồ cột cho versions
ver_labels = ['Total Discovered', 'Parsed Versions', 'Empty Versions']
ver_counts = [cov['total_versions_discovered'], cov['parsed_versions_count'], cov['empty_versions_count']]
bars = ax2.bar(ver_labels, ver_counts, color=['#1f77b4', '#2ca02c', '#ff7f0e'], width=0.55)
ax2.set_ylabel('Số lượng phiên bản')
ax2.set_title('Độ bao phủ Phiên bản (Versions)', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, yval + (max(ver_counts) * 0.015), f'{yval:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Hiệu quả Khử trùng lặp Xuyên phiên bản (Deduplication)

So sánh số lượng phần tử (`elements`) và mục tài liệu tham khảo (`bibtex`) **trước và sau khi khử trùng lặp** giữa các phiên bản.

In [ ]:
dedup_table = pd.DataFrame([
    {
        'Đối tượng': 'Elements (Cây phân cấp)',
        'Trước Deduplication': f"{dedup['elements_before_dedup']:,}",
        'Sau Deduplication': f"{dedup['elements_after_dedup']:,}",
        'Số lượng giảm': f"{dedup['elements_before_dedup'] - dedup['elements_after_dedup']:,}",
        'Tỷ lệ giảm (%)': f"{dedup['element_dedup_reduction_pct']} %"
    },
    {
        'Đối tượng': 'BibTeX References',
        'Trước Deduplication': f"{dedup['bibtex_entries_before_dedup']:,}",
        'Sau Deduplication': f"{dedup['bibtex_entries_after_dedup']:,}",
        'Số lượng giảm': f"{dedup['bibtex_entries_before_dedup'] - dedup['bibtex_entries_after_dedup']:,}",
        'Tỷ lệ giảm (%)': f"{dedup['bibtex_dedup_reduction_pct']} %"
    }
])
display(dedup_table)
display(Markdown(f"**Tổng số lệnh `\\cite{{}}` được đổi tên đồng bộ:** `{dedup['citations_renamed_count']:,}` lượt."))

# Grouped Bar Chart
import numpy as np

categories = ['Elements', 'BibTeX References']
before_counts = [dedup['elements_before_dedup'], dedup['bibtex_entries_before_dedup']]
after_counts = [dedup['elements_after_dedup'], dedup['bibtex_entries_after_dedup']]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(8.5, 4.8))
rects1 = ax.bar(x - width/2, before_counts, width, label='Trước Dedup', color='#d62728', alpha=0.85)
rects2 = ax.bar(x + width/2, after_counts, width, label='Sau Dedup', color='#2ca02c', alpha=0.85)

ax.set_ylabel('Số lượng bản ghi')
ax.set_title('So sánh Số lượng Bản ghi Trước và Sau khi Khử trùng lặp', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontweight='bold', fontsize=11)
ax.legend(frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.6)

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f'{h:,}', xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

for rect in rects2:
    h = rect.get_height()
    ax.annotate(f'{h:,}', xy=(rect.get_x() + rect.get_width() / 2, h),
                xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()